<a href="https://colab.research.google.com/github/SyahrialHipdi/Pes_Similiarity/blob/main/similiarity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving pes2021-all-players.csv to pes2021-all-players.csv


In [2]:
# ====================================
# LOAD DATA
# ====================================
import pandas as pd

df = pd.read_csv('pes2021-all-players.csv')

print(df.head())
print(df.columns)

             name shirt_number          team_name                  league  \
0        L. MESSI           10       FC BARCELONA          Spanish League   
1      C. RONALDO            7           JUVENTUS             Serie A TIM   
2  R. LEWANDOWSKI            9  FC BAYERN MÜNCHEN  Other European Leagues   
3          NEYMAR           10                PSG       Ligue 1 Uber Eats   
4    K. DE BRUYNE           17       MANCHESTER B          English League   

  nationality         region  height  weight  age        foot  ...  \
0   ARGENTINA  South America     170      72   33   Left foot  ...   
1    PORTUGAL         Europe     187      83   35  Right foot  ...   
2      POLAND         Europe     185      79   32  Right foot  ...   
3      BRAZIL  South America     175      68   28  Right foot  ...   
4     BELGIUM         Europe     181      68   29  Right foot  ...   

  skill_super_sub com_playing_style_trickster  com_playing_style_mazing_run  \
0               0                    

In [4]:
# ====================================
# MEMILIH FITUR
# ====================================
import numpy as np

# Kolom yang bukan atribut pemain
exclude_columns = [
    'Name',
    'Position',
    'Team'
]

feature_columns = [
    col for col in df.columns
    if col not in exclude_columns
    and np.issubdtype(df[col].dtype, np.number)
]

print(feature_columns)

['height', 'weight', 'age', 'LWF', 'SS', 'CF', 'RWF', 'LMF', 'DMF', 'CMF', 'AMF', 'RMF', 'LB', 'CB', 'RB', 'offensive_awareness', 'ball_control', 'dribbling', 'tight_possession', 'low_pass', 'lofted_pass', 'finishing', 'heading', 'place_kicking', 'curl', 'speed', 'acceleration', 'kicking_power', 'jump', 'physical_contact', 'balance', 'stamina', 'defensive_awareness', 'ball_winning', 'aggression', 'gk_awareness', 'gk_catching', 'gk_clearing', 'gk_reflexes', 'gk_reach', 'weak_foot_usage', 'weak_foot_accuracy', 'form', 'injury_resistance', 'overall_rating', 'rating_as_GK', 'rating_as_CB', 'rating_as_LB', 'rating_as_RB', 'rating_as_DMF', 'rating_as_CMF', 'rating_as_LMF', 'rating_as_RMF', 'rating_as_AMF', 'rating_as_LWF', 'rating_as_RWF', 'rating_as_SS', 'rating_as_CF', 'skill_cross_over_turn', 'skill_early_cross', 'skill_first_time_shot', 'skill_incisive_run', 'skill_one_touch_pass', 'skill_chip_shot_control', 'skill_heel_trick', 'skill_man_marking', 'skill_interception', 'skill_gk_penalty

In [5]:
# ====================================
# NORMALISASI
# ====================================
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X = scaler.fit_transform(df[feature_columns])

In [6]:
# ====================================
# KNN MODEL
# ====================================
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(
    n_neighbors=6,
    metric='euclidean'
)

knn.fit(X)

ValueError: Input X contains NaN.
NearestNeighbors does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
# ====================================
# CARI PEMAIN MIRIP
# ====================================
def pemain_mirip(nama_pemain):

    pemain = df[
        df['Name'].str.lower() == nama_pemain.lower()
    ]

    if pemain.empty:
        print("Pemain tidak ditemukan")
        return

    idx = pemain.index[0]

    distances, indices = knn.kneighbors(
        X[idx].reshape(1, -1)
    )

    hasil = []

    for i in range(1, len(indices[0])):

        pemain_idx = indices[0][i]

        hasil.append({
            "Nama": df.iloc[pemain_idx]["Name"],
            "Kemiripan": round(
                (1/(1+distances[0][i]))*100,
                2
            )
        })

    return pd.DataFrame(hasil)

In [ ]:
pemain_mirip("Kylian Mbappe")